# Bonus: RAG (adaptado a lab local)

**Qué se ve aquí:** cargar un PDF, partirlo en chunks, generar **embeddings**, guardarlos en un vector store y exponer la búsqueda como tool de un agente.

> **Requisito en el servidor Ollama:** `ollama pull nomic-embed-text` (modelo de embeddings, ~270 MB, corre bien en CPU).

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-2/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

## Semantic search

In [ ]:
# Sin cambios
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

In [ ]:
# Sin cambios
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: OpenAIEmbeddings es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain_openai import OpenAIEmbeddings
#
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: embeddings locales con Ollama. Requiere en el servidor: ollama pull nomic-embed-text
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url=OLLAMA_URL)
print(len(embeddings.embed_query("prueba")), "dimensiones")   # nomic-embed-text → 768

In [ ]:
# Sin cambios
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [ ]:
# Sin cambios
ids = vector_store.add_documents(documents=all_splits)

In [ ]:
# Sin cambios
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

## RAG Agent

In [ ]:
# Sin cambios
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     tools=[search_handbook],
#     system_prompt="You are a helpful agent that can search the employee handbook for information."
#     )

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → MODEL (gemma4 en Ollama, definido en la celda de setup).
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [ ]:
# Sin cambios
from pprint import pprint

pprint(response)